# ISIC2024 후속 검증 Image 결과 분석 17

이 notebook은 `.py` runner가 artifact를 만든 뒤에 후속 image baseline 결과를 읽고 비교하는 분석 문서다.

- 주 평가지표: `pauc_above_tpr80`
- 현재 풀에서는 `HAM10000`을 제외한다.
- `RETFound`는 로컬 checkpoint가 준비되어 있으므로 ready로 본다.
- 비교 anchor: `strict_base`, `strict_fe`, `strict_main_input` tabular baseline
- 실행 기준 원본: `.py` runner와 그 결과 artifact


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("/home/junkim2603a/proj/paper_ajou_dev").resolve()
os.chdir(ROOT)

IMAGE_MODELS = [
    "BioMedCLIP",
    "CheXzero",
    "DeiT-S",
    "DenseNet-121",
    "DINOv2 ViT-S",
    "EfficientNet-B0",
    "EyePACS",
    "MedCLIP",
    "MONET",
    "ResNet-50",
    "RETFound",
    "TorchXRayVision",
    "ViT-B_16",
]
IMAGE_CONFIG_ROOT = ROOT / "src" / "image_baselines"
IMAGE_LEADERBOARD_CANDIDATES = [
    ROOT / "artifacts" / "image_mlflow_leaderboard.csv",
]
EXPORT_DIR = ROOT / "artifacts" / "eda" / "isic2024" / "followup_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_json_sig(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8-sig"))


def find_primary_metric(columns: list[str], prefix: str) -> str | None:
    exact = [column for column in columns if column == prefix]
    if exact:
        return exact[0]
    candidates = sorted(column for column in columns if column.startswith(prefix))
    return candidates[0] if candidates else None


def has_torchxrayvision() -> bool:
    return importlib.util.find_spec("torchxrayvision") is not None


def image_readiness_table() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for model_name in IMAGE_MODELS:
        config_path = IMAGE_CONFIG_ROOT / model_name / "config.json"
        config = read_json_sig(config_path) if config_path.exists() else {}
        checkpoint_path = config.get("model", {}).get("checkpoint_path")
        checkpoint_exists = Path(checkpoint_path).exists() if checkpoint_path else None
        status = "ready"
        note = ""
        if model_name == "EyePACS":
            status = "ready"
            note = "efficientnet_b3 local checkpoint를 사용하며 5-class head 대신 2-class head로 fine-tuning 합니다"
        elif model_name == "TorchXRayVision":
            if has_torchxrayvision():
                status = "ready"
                note = "torchxrayvision 패키지가 설치되어 있어 현재 backend 설정으로 실행 가능합니다"
            else:
                status = "pending"
                note = "torchxrayvision 패키지만 설치되면 현재 backend 설정으로 실행할 수 있습니다"
        elif model_name == "MedCLIP":
            status = "ready"
            note = "공식 medclip package를 사용하며 첫 실행 시 pretrained weight를 로컬 cache에 저장합니다"
        elif model_name in {"CheXzero", "RETFound"} and not checkpoint_exists:
            status = "blocked"
            note = f"로컬 checkpoint가 없습니다: {checkpoint_path}"
        rows.append(
            {
                "model_name": model_name,
                "config_path": str(config_path),
                "checkpoint_path": checkpoint_path,
                "checkpoint_exists": checkpoint_exists,
                "status": status,
                "note": note,
            }
        )
    return pd.DataFrame(rows)


def load_parent_leaderboard() -> tuple[pd.DataFrame, Path | None]:
    for path in IMAGE_LEADERBOARD_CANDIDATES:
        if path.exists():
            return pd.read_csv(path), path
    return pd.DataFrame(), None


def load_trial_summaries() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for summary_path in sorted((ROOT / "artifacts").glob("*/*/summary.json")):
        if summary_path.parts[-3] not in IMAGE_MODELS:
            continue
        payload = read_json(summary_path)
        test_metrics = payload.get("test_metrics", {})
        val_metrics = payload.get("best_validation_metrics", {})
        rows.append(
            {
                "model_name": summary_path.parts[-3],
                "trial_name": summary_path.parts[-2],
                "summary_path": str(summary_path),
                "best_epoch": payload.get("best_epoch"),
                "best_val_pauc_above_tpr80": val_metrics.get("pauc_above_tpr80"),
                "test_pauc_above_tpr80": test_metrics.get("pauc_above_tpr80"),
                "test_average_precision": test_metrics.get("average_precision"),
                "test_auc_roc": test_metrics.get("auc_roc"),
            }
        )
    return pd.DataFrame(rows)


readiness_df = image_readiness_table()
display(readiness_df)


In [ ]:
leaderboard_df, leaderboard_path = load_parent_leaderboard()
trial_df = load_trial_summaries()

print("leaderboard 경로:", leaderboard_path)
print("leaderboard 행 수:", len(leaderboard_df))
print("trial 행 수:", len(trial_df))

if leaderboard_df.empty and trial_df.empty:
    display(
        Markdown(
            "아직 image 결과가 없습니다. runbook notebook에서 GPU 0/1 ready 모델 풀을 실행한 뒤 이 notebook을 다시 여세요."
        )
    )
else:
    if not leaderboard_df.empty:
        primary_column = find_primary_metric(list(leaderboard_df.columns), "best_pauc_above_tpr")
        visible = leaderboard_df.sort_values([primary_column], ascending=False) if primary_column else leaderboard_df.copy()
        display(visible)
    if not trial_df.empty:
        display(trial_df.sort_values(["test_pauc_above_tpr80", "test_auc_roc"], ascending=False))


In [ ]:
if trial_df.empty:
    display(Markdown("아직 trial 단위 image summary가 없어 비교 차트는 건너뜁니다."))
else:
    best_by_model = (
        trial_df.groupby("model_name", dropna=False)["test_pauc_above_tpr80"]
        .max()
        .sort_values(ascending=False)
        .reset_index()
    )
    display(best_by_model)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(best_by_model["model_name"], best_by_model["test_pauc_above_tpr80"])
    ax.set_title("Image 후속 검증 모델별 최고 Test pAUC@TPR>=0.80")
    ax.set_ylabel("test_pauc_above_tpr80")
    ax.set_xlabel("model_name")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
if leaderboard_df.empty:
    print("image leaderboard가 없어 export를 건너뜁니다.")
else:
    export_path = EXPORT_DIR / "image_followup_parent_leaderboard.csv"
    leaderboard_df.to_csv(export_path, index=False)
    print("저장 완료:", export_path)
